In [1]:
import os
import threading
import pandas as pd
import re
from flask import Flask, redirect, render_template, request, url_for
from textblob import TextBlob

# 1. Initialize Flask App and Folders
app = Flask(__name__)
UPLOAD_FOLDER = "uploads"
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
app.config["UPLOAD_FOLDER"] = UPLOAD_FOLDER

# Define a list of flirty keywords, phrases, and emojis
flirty_keywords = [
    "love", "miss you", "cute", "babe", "handsome", "beautiful", 
    "jaan", "sweetheart", "hot", "muah", "hug", "kiss", 
    "❤️", "💕", "😘", "😉", "😍", "🥰"
]

def detect_flirting(text):
    text_lower = str(text).lower()
    for word in flirty_keywords:
        if word in text_lower:
            return True
    return False

In [2]:
def analyze_chat(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    parsed_data = []
    for line in lines:
        if " - " in line:
            parsed_data.append(line)
        elif parsed_data:
            parsed_data[-1] += " " + line

    df = pd.DataFrame(parsed_data, columns=["RawText"])
    df[["Date_Time", "Message_Body"]] = df["RawText"].str.split(" - ", n=1, expand=True)
    df[["Date", "Time"]] = df["Date_Time"].str.split(r",\s*|\s+", n=1, expand=True, regex=True)
    df[["Name", "Chat"]] = df["Message_Body"].str.split(": ", n=1, expand=True)

    df = df.dropna(subset=["Name", "Chat"])
    df = df[~df["Chat"].str.contains("Messages and calls are end-to-end encrypted", na=False)]

    # Sentiment Analysis
    df["Sentiment_Score"] = df["Chat"].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
    df["Sentiment_Category"] = df["Sentiment_Score"].apply(
        lambda score: "Positive" if score > 0.05 else ("Negative" if score < -0.05 else "Neutral")
    )

    # Flirting Detection
    df["Is_Flirty"] = df["Chat"].apply(detect_flirting)

    # Statistics Calculations
    total_messages = len(df)
    participants = df["Name"].unique().tolist()
    sentiment_counts = df["Sentiment_Category"].value_counts().to_dict()
    flirty_counts = df[df["Is_Flirty"] == True]["Name"].value_counts().to_dict()

    return total_messages, sentiment_counts, participants, flirty_counts

In [3]:
from flask import render_template_string
@app.route("/", methods=["GET", "POST"])
def index():
    if request.method == "POST":
        if "file" not in request.files:
            return redirect(request.url)
        file = request.files["file"]
        if file.filename == "":
            return redirect(request.url)

        if file:
            file_path = os.path.join(app.config["UPLOAD_FOLDER"], file.filename)
            file.save(file_path)

            total_messages, sentiment_counts, participants, flirty_counts = analyze_chat(file_path)

            # Professional HTML template using Tailwind CSS
            html_template = """
            <!DOCTYPE html>
            <html lang="en">
            <head>
                <meta charset="UTF-8">
                <meta name="viewport" content="width=device-width, initial-scale=1.0">
                <title>WhatsApp Chat Analytics Dashboard</title>
                <script src="https://cdn.tailwindcss.com"></script>
            </head>
            <body class="bg-slate-50 text-slate-800 font-sans antialiased">
                <div class="max-w-4xl mx-auto px-4 py-10">
                    <!-- Header -->
                    <div class="mb-8 text-center">
                        <h1 class="text-3xl font-extrabold text-slate-900 tracking-tight">WhatsApp Chat Analytics</h1>
                        <p class="text-sm text-slate-500 mt-1">Detailed sentiment and interaction breakdown</p>
                    </div>

                    <!-- Metrics Overview Grid -->
                    <div class="grid grid-cols-1 md:grid-cols-2 gap-6 mb-8">
                        <div class="bg-white p-6 rounded-2xl shadow-sm border border-slate-200">
                            <h3 class="text-xs font-semibold uppercase tracking-wider text-slate-400 mb-1">Total Messages</h3>
                            <p class="text-3xl font-bold text-indigo-600">{{ total_messages }}</p>
                        </div>
                        <div class="bg-white p-6 rounded-2xl shadow-sm border border-slate-200">
                            <h3 class="text-xs font-semibold uppercase tracking-wider text-slate-400 mb-1">Participants</h3>
                            <p class="text-lg font-medium text-slate-700">{{ participants | join(', ') }}</p>
                        </div>
                    </div>

                    <!-- Breakdowns Grid -->
                    <div class="grid grid-cols-1 md:grid-cols-2 gap-6 mb-8">
                        <!-- Sentiment Breakdown -->
                        <div class="bg-white p-6 rounded-2xl shadow-sm border border-slate-200">
                            <h3 class="text-lg font-semibold text-slate-900 mb-4 pb-2 border-b border-slate-100">Sentiment Breakdown</h3>
                            <ul class="space-y-3">
                                {% for sentiment, count in sentiment_counts.items() %}
                                <li class="flex justify-between items-center text-sm">
                                    <span class="font-medium text-slate-600">{{ sentiment }}</span>
                                    <span class="bg-slate-100 text-slate-700 px-2.5 py-1 rounded-full font-semibold">{{ count }}</span>
                                </li>
                                {% endfor %}
                            </ul>
                        </div>

                        <!-- Flirty Messages Breakdown -->
                        <div class="bg-white p-6 rounded-2xl shadow-sm border border-slate-200">
                            <h3 class="text-lg font-semibold text-slate-900 mb-4 pb-2 border-b border-slate-100">Flirty Messages Breakdown</h3>
                            {% if flirty_counts %}
                            <ul class="space-y-3">
                                {% for name, count in flirty_counts.items() %}
                                <li class="flex justify-between items-center text-sm">
                                    <span class="font-medium text-slate-600">{{ name }}</span>
                                    <span class="bg-pink-50 text-pink-600 px-2.5 py-1 rounded-full font-semibold">{{ count }} messages</span>
                                </li>
                                {% endfor %}
                            </ul>
                            {% else %}
                            <p class="text-sm text-slate-400 italic">No flirty messages detected.</p>
                            {% endif %}
                        </div>
                    </div>

                    <!-- Actions -->
                    <div class="text-center">
                        <a href="/" class="inline-block bg-indigo-600 hover:bg-indigo-700 text-white font-medium px-6 py-2.5 rounded-xl transition shadow-sm">Analyze Another Chat</a>
                    </div>
                </div>
            </body>
            </html>
            """
            return render_template_string(
                html_template, 
                total_messages=total_messages, 
                participants=participants, 
                sentiment_counts=sentiment_counts, 
                flirty_counts=flirty_counts
            )

    # Professional Upload Page UI
    upload_template = """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Upload WhatsApp Chat</title>
        <script src="https://cdn.tailwindcss.com"></script>
    </head>
    <body class="bg-slate-50 text-slate-800 font-sans antialiased flex items-center justify-center min-h-screen">
        <div class="bg-white p-8 rounded-2xl shadow-sm border border-slate-200 max-w-md w-full text-center">
            <h2 class="text-2xl font-bold text-slate-900 mb-2">WhatsApp Chat Analyzer</h2>
            <p class="text-sm text-slate-500 mb-6">Upload your exported chat `.txt` file to view deep insights.</p>
            
            <form method="POST" enctype="multipart/form-data" class="space-y-4">
                <div class="border-2 border-dashed border-slate-200 rounded-xl p-6 hover:border-indigo-500 transition cursor-pointer">
                    <input type="file" name="file" accept=".txt" class="block w-full text-sm text-slate-500 file:mr-4 file:py-2 file:px-4 file:rounded-full file:border-0 file:text-sm file:font-semibold file:bg-indigo-50 file:text-indigo-600 hover:file:bg-indigo-100">
                </div>
                <button type="submit" class="w-full bg-indigo-600 hover:bg-indigo-700 text-white font-medium py-2.5 rounded-xl transition shadow-sm">Analyze Chat</button>
            </form>
        </div>
    </body>
    </html>
    """
    return render_template_string(upload_template)

In [4]:
def run_flask():
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask)
flask_thread.start()
print("Flask app is running! Go to http://127.0.0.1:5000/ in your browser.")

Flask app is running! Go to http://127.0.0.1:5000/ in your browser.
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [18/Sep/2026 18:55:10] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [18/Sep/2026 18:55:26] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [18/Sep/2026 18:55:42] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [18/Sep/2026 18:56:58] "POST / HTTP/1.1" 200 -
